# JS13 Praktikum 3 - Regresi Harga Rumah dengan Neural Network

Praktikum ini membandingkan dua learning rate pada regresi harga rumah. Dataset dibuat secara sintetis agar notebook dapat dijalankan tanpa mengunduh data eksternal. Pendekatan ini juga menghindari `load_boston` yang sudah dihapus dari scikit-learn versi baru.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
n_samples = 300
data = pd.DataFrame({
    "luas_m2": rng.uniform(40, 250, n_samples),
    "jumlah_kamar": rng.integers(1, 6, n_samples),
    "umur_rumah": rng.uniform(0, 30, n_samples),
    "jarak_pusat_km": rng.uniform(1, 25, n_samples),
})
data["harga_juta"] = (
    2.8 * data["luas_m2"]
    + 45 * data["jumlah_kamar"]
    - 2.5 * data["umur_rumah"]
    - 8 * data["jarak_pusat_km"]
    + rng.normal(0, 25, n_samples)
)

X = data.drop(columns="harga_juta")
y = data[["harga_juta"]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)
y_train_scaled = y_scaler.fit_transform(y_train).ravel()
y_test_scaled = y_scaler.transform(y_test).ravel()
print(data.head())

      luas_m2  jumlah_kamar  umur_rumah  jarak_pusat_km  harga_juta
0  202.530770             4    1.270517        7.248365  660.396816
1  132.164472             4   26.524371        8.739771  415.135763
2  220.305563             4   21.287349        6.819588  682.234235
3  186.447286             1    5.193835       12.516722  445.513241
4   59.777243             5    2.751630       17.398201  261.804150


## Eksperimen learning rate

Model menggunakan dua hidden layer ReLU dan output linear. Nilai loss yang dibandingkan adalah MSE pada data target yang sudah dinormalisasi. MAE, RMSE, dan R² dihitung kembali dalam satuan juta rupiah.

In [2]:
def run_regression(name, learning_rate_init):
    model = MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        solver="adam",
        learning_rate_init=learning_rate_init,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.2,
        n_iter_no_change=30,
        random_state=42,
    )
    model.fit(X_train_scaled, y_train_scaled)
    prediction_scaled = model.predict(X_test_scaled)
    prediction = y_scaler.inverse_transform(prediction_scaled.reshape(-1, 1)).ravel()
    actual = y_test.to_numpy().ravel()
    return {
        "Konfigurasi": name,
        "Learning rate": learning_rate_init,
        "Loss akhir (MSE scaled)": model.loss_,
        "MAE (juta)": mean_absolute_error(actual, prediction),
        "RMSE (juta)": np.sqrt(mean_squared_error(actual, prediction)),
        "R2": r2_score(actual, prediction),
        "predictions": prediction,
        "model": model,
    }


lr_001 = run_regression("Learning rate 0.001", 0.001)
lr_01 = run_regression("Learning rate 0.01", 0.01)
results = pd.DataFrame([
    {k: v for k, v in result.items() if k not in {"predictions", "model"}}
    for result in [lr_001, lr_01]
])
results.style.format({
    "Learning rate": "{:.3f}",
    "Loss akhir (MSE scaled)": "{:.6f}",
    "MAE (juta)": "{:.2f}",
    "RMSE (juta)": "{:.2f}",
    "R2": "{:.4f}",
})

,Konfigurasi,Learning rate,Loss akhir (MSE scaled),MAE (juta),RMSE (juta),R2
0,Learning rate 0.001,0.001,0.010901,25.20,32.23,0.9714
1,Learning rate 0.01,0.010,0.006775,21.79,27.71,0.9789


In [3]:
comparison = pd.DataFrame({
    "Harga aktual (juta)": y_test.to_numpy().ravel()[:10],
    "Prediksi lr=0.001": lr_001["predictions"][:10],
    "Prediksi lr=0.01": lr_01["predictions"][:10],
})
print(comparison.round(2).to_string(index=False))
best = results.loc[results["RMSE (juta)"].idxmin()]
print(f"\nLearning rate dengan RMSE terkecil: {best['Konfigurasi']}")


 Harga aktual (juta)  Prediksi lr=0.001  Prediksi lr=0.01
              334.94             316.92            332.52
              269.79             325.67            312.88
              200.09             177.55            174.32
              404.93             480.26            470.27
              371.19             399.14            413.01
              115.65             113.15            145.14
              532.51             570.97            535.62
              628.86             583.58            621.11
              763.57             737.99            745.22
              314.21             250.93            274.87

Learning rate dengan RMSE terkecil: Learning rate 0.01


## Kesimpulan

Learning rate mengatur besar langkah pembaruan bobot. Learning rate yang terlalu kecil dapat membuat proses belajar lambat, sedangkan nilai yang terlalu besar dapat membuat loss berosilasi atau tidak stabil. Konfigurasi terbaik pada eksperimen ini dipilih berdasarkan RMSE testing terkecil.